## Anna Giczewska ST554 - Final Project ¶
Date 5/2/2026

**Short Project Description:** In this project, I use PySpark to build an elastic net regression model for predicting Power_Zone_3 based on the other available variables in the power consumption data set. 

After fitting the model with 5-fold cross-validation, I use the fitted model in a structured streaming example. The streaming part reads small CSV files from a watched folder, makes predictions on the incoming data, calculates residuals, and writes the results to the console.


### 1. Import packages and start Spark


In [1]:
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, PCA

from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

In [2]:
# start a Spark session for this project
spark = SparkSession.builder.appName("final_project").getOrCreate()
# reduce the amount of Spark log output
spark.sparkContext.setLogLevel("WARN")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 09:23:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Read and inspect the modeling data

In [3]:
# read the modeling data with pandas first
power_pd = pd.read_csv("power_ml_data.csv")

print(power_pd.shape)
power_pd.head()

(47174, 10)


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
# convert pandas data frame to spark data frame
power_df = spark.createDataFrame(power_pd)

power_df.printSchema()
power_df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

In [5]:
# just to see the columns clearly
print(power_df.columns)

['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows', 'Power_Zone_1', 'Power_Zone_2', 'Power_Zone_3', 'Month', 'Hour']


### 3. Build the preprocessing pipeline

In this section, I create the transformations required in the assignment:
- cast `Hour` to double
- create a binary hour variable
- one-hot encode `Month`
- apply PCA to the weather variables
- assemble all predictors into a final `features` vector

In [6]:
# cast Hour to double and create label
sql_stage = SQLTransformer(
    statement="""
    SELECT *,
           CAST(Hour AS DOUBLE) AS Hour_double,
           Power_Zone_3 AS label
    FROM __THIS__
    """
)

# binary hour variable
hour_bin = Binarizer(
    threshold=6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

# one-hot encode Month
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index",
    handleInvalid="keep"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_vec"]
)

In [7]:
# put weather variables together for PCA
weather_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="weather_features"
)

# PCA with 2 components
pca_stage = PCA(
    k=2,
    inputCol="weather_features",
    outputCol="pca_features"
)

In [8]:
# final feature vector
final_assembler = VectorAssembler(
    inputCols=[
        "pca_features",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_vec"
    ],
    outputCol="features"
)

feature_pipeline = Pipeline(stages=[
    sql_stage,
    hour_bin,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca_stage,
    final_assembler
])

feature_model = feature_pipeline.fit(power_df)
feature_df = feature_model.transform(power_df)

feature_df.select("label", "features").show(5, truncate=False)

+-----------+-------------------------------------------------------------------------------------+
|label      |features                                                                             |
+-----------+-------------------------------------------------------------------------------------+
|20240.96386|(17,[0,1,3,4,8],[1.794404863656954,-0.7412746447404517,34055.6962,16128.87538,1.0])  |
|20131.08434|(17,[0,1,3,4,8],[1.8060408300982311,-0.7108534239558543,29814.68354,19375.07599,1.0])|
|19668.43373|(17,[0,1,3,4,8],[1.8102297630563902,-0.7283113191159,29128.10127,19006.68693,1.0])   |
|18899.27711|(17,[0,1,3,4,8],[1.7986676517408842,-0.7220913032200007,28228.86076,18361.09422,1.0])|
|18442.40964|(17,[0,1,3,4,8],[1.863287201637971,-0.7323046647776624,27335.6962,17872.34043,1.0])  |
+-----------+-------------------------------------------------------------------------------------+
only showing top 5 rows


### 4. Fit the elastic net model with cross-validation

Here I fit a linear regression model with elastic net regularization. I use 5-fold cross-validation and RMSE as the evaluation metric.

In [9]:
# define the linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# put all preprocessing steps and the model into one pipeline
full_pipeline = Pipeline(stages=[
    sql_stage,
    hour_bin,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca_stage,
    final_assembler,
    lr
])

In [10]:
# create the tuning grid for regParam and elasticNetParam
reg_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]
enet_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, reg_values)
    .addGrid(lr.elasticNetParam, enet_values)
    .build()
)

In [11]:
# use RMSE to evaluate each model during cross-validation
rmse_eval = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# use 5-fold cross-validation
cv = CrossValidator(
    estimator=full_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rmse_eval,
    numFolds=5
)

In [12]:
# fit the cross-validated model
cv_model = cv.fit(power_df)

# extract the best fitted model
best_model = cv_model.bestModel
best_lr = best_model.stages[-1]

# print the best tuning values and best CV RMSE
print("best regParam:", best_lr.getRegParam())
print("best elasticNetParam:", best_lr.getElasticNetParam())
print("best CV RMSE:", min(cv_model.avgMetrics))

26/05/03 09:23:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/03 09:23:33 WARN Instrumentation: [d0cb71ae] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 09:23:35 WARN Instrumentation: [8291dfa0] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 09:23:37 WARN Instrumentation: [99db074a] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 09:23:38 WARN Instrumentation: [be234bb5] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 09:23:38 WARN Instrumentation: [be234bb5] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/05/03 09:23:39 WARN Instrumentation: [74af2a74] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 09:23:40 WARN Instrumentatio

best regParam: 1.0
best elasticNetParam: 0.75
best CV RMSE: 2148.134119073199


### 5. Evaluate model performance on the training data

After choosing the best model from cross-validation, I apply it to the full training data. Then I compute the training RMSE and create residuals using label - prediction.

In [13]:
# use the best model to make predictions on the full training data
train_pred = best_model.transform(power_df)

# calculate training RMSE
train_rmse = rmse_eval.evaluate(train_pred)
print("training RMSE:", train_rmse)

training RMSE: 2147.0996228240892


In [14]:
# create a residual column and keep only the main output columns
resid_df = (
    train_pred
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)

# display the first 20 residual results
resid_df.show(20, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20876.083229036987|-635.1193690369873|
|20131.08434|18654.705893205988|1476.3784467940131|
|19668.43373|18199.36076983389 |1469.0729601661114|
|18899.27711|17585.53225859213 |1313.7448514078678|
|18442.40964|16992.360819066245|1450.0488209337564|
|18130.12048|16512.928183863707|1617.1922961362943|
|17945.06024|16088.662179293993|1856.3980607060057|
|17459.27711|15718.238693985839|1741.0384160141602|
|17025.54217|15266.762470118347|1758.7796998816539|
|16794.21687|14934.174881278172|1860.0419887218286|
|16638.07229|14648.474133621037|1989.5981563789628|
|16395.18072|14411.079355731214|1984.1013642687867|
|16117.59036|14079.011459027908|2038.578900972092 |
|15822.6506 |13621.195031278   |2201.455568722    |
|15672.28916|13446.802350791057|2225.4868092089437|
|15597.10843|13298.882537420352|2298.2258925796486|
|15510.36145

### 6. Set up the streaming folders and schema

Next, I prepare the folder that Spark will watch for incoming CSV files. I also create a checkpoint folder and define the schema for the streaming data

In [28]:
# os is used to create folders for streaming
import os

# create the folder Spark will watch and the checkpoint folder
os.makedirs("stream_data", exist_ok=True)
os.makedirs("checkpoint_folder", exist_ok=True)

In [29]:
# use the training data schema for the incoming stream
stream_schema = power_df.schema
stream_schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

### 7. Read the CSV stream

In this step, Spark continuously watches the stream_data folder for new CSV files.

In [30]:
# create a streaming DataFrame from the watched folder
stream_df = (
    spark.readStream
    .option("header", True)
    .schema(stream_schema)
    .csv("stream_data")
)

# confirm that this is a streaming DataFrame
print(stream_df.isStreaming)

True


### 8. Predict on the stream and join two stream transformations

The assignment asks for two transformations of the same stream: a prediction stream with residuals and another stream where Power_Zone_3 is renamed to label. Then these two results are joined using label.

In [31]:
# first stream transformation: make predictions and calculate residuals
pred_stream = (
    best_model
    .transform(stream_df)
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)

# second stream transformation: rename Power_Zone_3 to label
label_stream = (
    stream_df
    .withColumnRenamed("Power_Zone_3", "label")
    .select("label")
)

# join the two stream transformations using the label column
joined_stream = (
    pred_stream
    .join(label_stream, on="label", how="inner")
    .select("label", "prediction", "residual")
)

In [32]:
# write the streaming results to the console
query = (
    joined_stream.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "checkpoint_folder")
    .start()
)

26/05/03 10:00:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|13324.33735|11941.579973000551|1382.7573769994488|
|13253.25228|9865.361112352757 |3387.8911676472435|
|20130.8502 |20489.444336738634|-358.5941367386331|
|13802.8192 |12625.727027331404|1177.0921726685956|
|23701.66154|21234.605869670184|2467.055670329817 |
+-----------+------------------+------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|14833.73494|16612.374490225488|-1778.6395502254873|
|41615.39749|38683.733422462225|2931.6640675377785 |
|11100.55385|12931.748791686508|-1831.1949416865082|
|16779.89785|17605.667899814645|-825.7700498146442 |
|11356.76113|12708.329053080968|-1351.5679230809674|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|13404.9848 |13768.527816894455|-363.5430168944549 |
|13532.17569|13663.069541397917|-130.8938513979174 |
|7133.733493|7758.765135754349 |-625.0316427543494 |
|10912.10031|15771.828228487853|-4859.7279184878535|
|21703.82445|23185.8564045355  |-1482.0319545355014|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|14173.48315|13548.551809566625|624.9313404333752  |
|15350.78031|15324.684634521893|26.095675478107296 |
|18792.36923|20202.19103386969 |-1409.8218038696905|
|13633.61345|12407.157554087384|1226.455895912617  |
|26293.55649|26258.4874596758  |35.06903032419723  |
+-----------+------------------+-------------------+



...

-------------------------------------------
Batch: 11
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|17250.90909|19326.236364584653|-2075.3272745846516|
|23328.90282|25434.12421814526 |-2105.2213981452624|
|27528.70293|27867.427701083892|-338.72477108389285|
|15167.41641|14168.806198568995|998.6102114310052  |
|25954.97976|26258.11461017538 |-303.13485017538187|
|9444.417767|6984.16786719702  |2460.2498998029805 |
|9478.991597|9192.491510563568 |286.50008643643196 |
|14544.57831|12312.54247081142 |2232.03583918858   |
|10808.02432|8738.29310452767  |2069.7312154723313 |
|12896.03842|11430.463313856335|1465.5751061436658 |
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 12
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|23928.3871 |22165.04302458907 |1763.3440754109288 |
|12390.76609|11280.967302085539|1109.7987879144603 |
|14903.04392|14902.813621554275|0.23029844572556613|
|26368.0    |23325.846349344447|3042.153650655553  |
|18548.36364|19426.735404143918|-878.371764143918  |
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|16377.83133|19233.859113262195|-2856.027783262194 |
|10274.18968|8716.777585157712 |1557.4120948422878 |
|24950.32258|24242.87592315876 |707.446656841239   |
|17793.55623|17460.86366186489 |332.69256813510947 |
|11670.51256|13085.734297367024|-1415.2217373670246|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|16110.63317|16801.37576834694 |-690.742598346942  |
|27193.10769|27076.020698942153|117.08699105784763 |
|11281.93548|12904.206846077872|-1622.2713660778718|
|17834.30962|22556.437468602315|-4722.127848602315 |
|15289.10769|15965.30229193553 |-676.1946019355291 |
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|16048.01921|16309.084741639623|-261.06553163962235|
|13778.70968|12860.153191787236|918.5564882127637  |
|11288.6747 |12843.582439561427|-1554.9077395614277|
|12173.07457|13803.096799672809|-1630.022229672808 |
|21349.62814|20027.490732229933|1322.1374077700675 |
+-----------+------------------+-------------------+



### Final Summary

In this project, I used PySpark to build an elastic net regression model to predict `Power_Zone_3` from the other available variables in the power consumption data.

To prepare the data, I: cast Hour as a numeric variable, created a binary hour variable, one-hot encoded Month, used PCA on the weather-related variables, combined the predictors into a final feature vector. 

Then I fit a linear regression model with elastic net regularization using 5-fold cross-validation and RMSE as the evaluation metric. After fitting the model, I examined the best tuning parameters, the cross-validation RMSE, the training RMSE, and the residuals.

In the streaming part of the project, I created a structured streaming pipeline that reads CSV files from a folder, applies the fitted model to the incoming data, calculates predictions and residuals, and writes the results to the console.

The separate file stream_power_data.py was used to simulate streaming data by writing small CSV batches into the watched folder over time.